# Running the Weather MLOps project

This notebook contains the main commands needed to prepare and run the project on Windows. Run all commands from the repository root unless another directory is specified.

## 1. Requirements

Install these tools before starting:

- Python 3.12
- Git
- Docker Desktop

Docker Desktop must be running before using Docker or Airflow.

## 2. Create the virtual environment

Open PowerShell in the repository root and run:

```powershell
python -m venv .venv
Set-ExecutionPolicy -Scope Process -ExecutionPolicy Bypass
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
pip install -r requirements.txt
```

After activation, the terminal prompt should start with `(.venv)`.

In [ ]:
import sys
print(sys.version)
print(sys.executable)

## 3. Run the data and model pipeline

The DVC pipeline downloads historical weather, prepares train and test datasets, trains the model and saves the metrics.

```powershell
dvc repro
dvc dag
dvc metrics show
dvc status
```

Important outputs:

```text
data/raw/weather_history.csv
data/processed/train.csv
data/processed/test.csv
models/model.pkl
metrics/metrics.json
```

In [ ]:
import json
from pathlib import Path

metrics_path = Path('../metrics/metrics.json')
if not metrics_path.exists():
    metrics_path = Path('metrics/metrics.json')

with metrics_path.open(encoding='utf-8') as file:
    metrics = json.load(file)

metrics

## 4. View MLflow results

Training writes model parameters, metrics and artifacts to MLflow. Start the MLflow server in a separate PowerShell terminal:

```powershell
cd D:\IU\PMLDL\weather_prediction
.\.venv\Scripts\mlflow.exe server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
```

Open `http://127.0.0.1:5000` and select the `weather-temperature-forecast` experiment.

Stop MLflow with `Ctrl+C` in its terminal.

## 5. Run FastAPI and Streamlit with Docker

Make sure `models/model.pkl` exists, then run:

```powershell
docker compose -f code/deployment/docker-compose.yml up --build -d
```

Open the services:

- Streamlit: `http://127.0.0.1:8501`
- FastAPI documentation: `http://127.0.0.1:8000/docs`
- FastAPI health check: `http://127.0.0.1:8000/health`

Check the containers:

```powershell
docker compose -f code/deployment/docker-compose.yml ps
```

Stop them:

```powershell
docker compose -f code/deployment/docker-compose.yml down
```

## 6. Run the automated Airflow pipeline

Stop the deployment Compose services first, because Airflow creates its own API and Streamlit containers.

```powershell
docker compose -f code/deployment/docker-compose.yml down
docker compose -f services/airflow/docker-compose.yml up --build -d
docker logs weather-airflow --tail 100
```

Open `http://127.0.0.1:8080`. Login credentials are printed in the Airflow container logs.

Find `weather_mlops_pipeline`, enable it and choose `Single Run` for the first test. The task order is:

```text
run_dvc_pipeline -> deploy_application -> verify_services
```

After the first test, Airflow runs the DAG automatically every five minutes. Successful tasks are shown in green.

## 7. Stop everything

Stop Airflow:

```powershell
docker compose -f services/airflow/docker-compose.yml down
```

Remove the API and Streamlit containers created by Airflow:

```powershell
docker rm -f weather-api weather-app
```

If a container is already absent, Docker may print `No such container`. This is safe.

Stop MLflow with `Ctrl+C` in the MLflow terminal and leave the Python environment with:

```powershell
deactivate
```

## 8. Final check

Before committing changes, check the repository:

```powershell
dvc status
git status
```

Generated datasets, the model, MLflow files and Airflow logs should remain ignored by Git.